# Notebook 3 - Chatbot Models

This notebook trains and evaluates the two chatbot architectures on the QA dataset built in notebook 1:

1. **LSTM seq2seq with Bahdanau attention** — bidirectional LSTM encoder, LSTM decoder, additive attention.
2. **Transformer** — from-scratch encoder-decoder with multi-head self-attention, sinusoidal positional encoding, warmup + inverse-sqrt LR schedule.

Both models share a single `WordTokenizer` fit on the union of questions and answers, and both are trained on the same ~2k QA pairs for a direct comparison. All code lives in `task2_chatbot/models/lstm_chatbot.py`, `task2_chatbot/models/transformer_chatbot.py`, `task2_chatbot/train.py`, and `task2_chatbot/evaluate.py` — this notebook is the narrated wrapper.

> **Runtime note**: the default is 30 epochs, which is slow on CPU (~15–25 minutes for both models). Drop `EPOCHS` to 5–10 for a quick demo run.

## Setup

In [ ]:
import os
import sys
import json
import torch
import matplotlib.pyplot as plt

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), os.pardir, os.pardir))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EPOCHS = 30  # drop to 5-10 for a quick demo

print(f"Device: {DEVICE}")
print(f"Epochs: {EPOCHS}")

In [ ]:
qa_path = os.path.join(PROJECT_ROOT, "task2_chatbot", "data", "qa_pairs.json")
with open(qa_path, "r", encoding="utf-8") as f:
    qa_pairs = json.load(f)

print(f"Loaded {len(qa_pairs):,} QA pairs")

## Shared tokenizer

Both chatbots use the **same vocabulary**, so we fit a single `WordTokenizer` over all questions + answers. The tokenizer also reserves four special tokens: `<PAD>` (0), `<UNK>` (1), `<SOS>` (2), `<EOS>` (3).

In [ ]:
from task2_chatbot.train import build_tokenizer

tokenizer = build_tokenizer(qa_pairs)
print(f"Vocabulary size: {tokenizer.vocab_size:,}")
print(f"Special tokens:  PAD={tokenizer.pad_idx}, UNK={tokenizer.unk_idx}, "
      f"SOS={tokenizer.sos_idx}, EOS={tokenizer.eos_idx}")

sample_q = qa_pairs[0]["question"]
sample_a = qa_pairs[0]["answer"]
print(f"\nExample question: {sample_q!r}")
print(f"  encoded (with specials): {tokenizer.encode(sample_q, add_special=True)}")
print(f"Example answer:   {sample_a!r}")
print(f"  encoded (with specials): {tokenizer.encode(sample_a, add_special=True)}")

## Part 1 - LSTM seq2seq chatbot

**Architecture** (`task2_chatbot/models/lstm_chatbot.py`):

- **Encoder**: embedding (dim 128) → 2-layer bidirectional LSTM (hidden 256) → linear projections to merge the forward+backward hidden and cell states.
- **Attention**: Bahdanau-style additive attention over encoder outputs.
- **Decoder**: embedding → LSTM with concatenated `[embedded ⊕ context]` input → linear output over vocabulary (uses `[output ⊕ context ⊕ embedded]` for prediction).
- **Training**: teacher-forcing ratio decreasing from 1.0 → 0.5 over epochs, gradient clip at 1.0, Adam + StepLR.

In [ ]:
from task2_chatbot.train import train_lstm_chatbot

lstm_chatbot, lstm_stats = train_lstm_chatbot(
    qa_pairs, tokenizer,
    batch_size=32, epochs=EPOCHS, lr=0.001, device=DEVICE,
)

In [ ]:
print("--- LSTM chatbot sample responses ---\n")
for q in ["What is the capital of France?",
         "How does gravity work?",
         "Hello!",
         "What is a computer?"]:
    print(f"Q: {q}")
    print(f"A: {lstm_chatbot.respond(tokenizer, q, max_len=30, device=DEVICE)}")
    print("-" * 60)

## Part 2 - Transformer chatbot

**Architecture** (`task2_chatbot/models/transformer_chatbot.py`):

- **Embeddings**: shared token embeddings scaled by `√d_model` (128), plus sinusoidal positional encoding.
- **Encoder**: 3 layers, each with multi-head self-attention (4 heads) + residual/LayerNorm + position-wise feed-forward (d_ff=512).
- **Decoder**: 3 layers, each with masked self-attention, cross-attention over encoder outputs, and feed-forward.
- **Masks**: source padding mask + target causal (lower-triangular) mask to prevent the decoder from peeking ahead.
- **Training**: Adam (β1=0.9, β2=0.98), Noam-style LR schedule (warmup 400 steps then inverse-sqrt decay), no teacher-forcing variable (target-shifted one step, standard transformer-style).

In [ ]:
from task2_chatbot.train import train_transformer_chatbot

trans_chatbot, trans_stats = train_transformer_chatbot(
    qa_pairs, tokenizer,
    batch_size=32, epochs=EPOCHS, lr=0.0005, device=DEVICE,
)

In [ ]:
print("--- Transformer chatbot sample responses ---\n")
for q in ["What is the capital of France?",
         "How does gravity work?",
         "Hello!",
         "What is a computer?"]:
    print(f"Q: {q}")
    print(f"A: {trans_chatbot.respond(tokenizer, q, max_len=30, device=DEVICE)}")
    print("-" * 60)

## Part 3 - Training loss curves

Side-by-side per-epoch training loss. Transformer typically drops faster thanks to parallel attention; LSTM starts higher and catches up more slowly.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5))

ax1.plot(range(1, len(lstm_stats["losses"]) + 1), lstm_stats["losses"],
         "b-o", markersize=3, label="LSTM chatbot")
ax1.set_title("LSTM chatbot training loss")
ax1.set_xlabel("epoch"); ax1.set_ylabel("cross-entropy")
ax1.grid(alpha=0.3); ax1.legend()

ax2.plot(range(1, len(trans_stats["losses"]) + 1), trans_stats["losses"],
         "r-o", markersize=3, label="Transformer chatbot")
ax2.set_title("Transformer chatbot training loss")
ax2.set_xlabel("epoch"); ax2.set_ylabel("cross-entropy")
ax2.grid(alpha=0.3); ax2.legend()

plt.tight_layout()
plt.show()

print(f"\nLSTM         final loss: {lstm_stats['final_loss']:.4f}")
print(f"Transformer  final loss: {trans_stats['final_loss']:.4f}")

## Part 4 - Comparison across 10 test questions

We reuse `evaluate_chatbots` from `task2_chatbot/evaluate.py`. It runs a fixed set of 10 questions through both models, prints a perf summary (time / memory / params / final loss), and a side-by-side response table. It also writes `chatbot_loss_curves.png` to the project root.

In [ ]:
from task2_chatbot.evaluate import evaluate_chatbots

chatbot_results = {
    "lstm_chatbot":        (lstm_chatbot,  lstm_stats),
    "transformer_chatbot": (trans_chatbot, trans_stats),
}

report = evaluate_chatbots(tokenizer, chatbot_results, device=DEVICE)

## Part 5 - Ask your own question

Edit `MY_QUESTION` and re-run this cell to compare both models' responses on any prompt.

In [ ]:
MY_QUESTION = "What is photosynthesis?"

print(f"Q: {MY_QUESTION}\n")
print("LSTM chatbot:")
print(" ", lstm_chatbot.respond(tokenizer, MY_QUESTION, max_len=40, device=DEVICE))
print("\nTransformer chatbot:")
print(" ", trans_chatbot.respond(tokenizer, MY_QUESTION, max_len=40, device=DEVICE))

## Takeaways

- **LSTM seq2seq** learns sequentially; the Bahdanau attention gives it a soft alignment between the question and answer tokens. Shorter/simpler responses, sometimes generic.
- **Transformer** parallelizes over the sequence and benefits from multi-head attention + positional encoding. Reaches lower training loss per epoch and handles longer dependencies better, but the tiny dataset (~2k pairs) limits its advantage.
- Both models are trained **from scratch** with no pretrained embeddings, which is the main ceiling on quality. In a real application you would seed with pretrained token embeddings or fine-tune a small pretrained encoder.

Next: [`04_summary.ipynb`](../../04_summary.ipynb) pulls together the full evaluation report, parameter counts, and cross-architecture conclusions.